# EZHit fine-tuning notebook

This notebook fine-tunes EZHit on a user-provided enzyme–reaction dataset.




## 1. Setup

This cell clones the EZHit GitHub repository and installs only the missing packages needed for this notebook.

It intentionally avoids reinstalling core numerical packages such as NumPy, SciPy, pandas, or scikit-learn.

In [ ]:
#@title Setup environment { display-mode: "form" }
# Stable Colab setup for EZHit
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/ld139/EzHit.git"
REPO_DIR = Path("/content/EzHit")

if REPO_DIR.exists():
    print(f"Repository already exists: {REPO_DIR}")
else:
    subprocess.check_call(["git", "clone", "-q", REPO_URL, str(REPO_DIR)])
    print(f"Cloned repository to: {REPO_DIR}")

sys.path.insert(0, str(REPO_DIR))

print("Python version:", sys.version)

def run_pip(args, title):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)
    cmd = [sys.executable, "-m", "pip"] + args
    print(" ".join(cmd))
    subprocess.check_call(cmd)

# Do not upgrade/reinstall numpy, scipy, sklearn, or pandas here.
# Recent Colab runtimes already provide a working numerical stack.
run_pip(["install", "-q", "huggingface_hub>=0.23", "transformers>=4.38,<4.47", "sentencepiece", "ipywidgets"], 
        "Installing HuggingFace / transformer utilities")

# Use maintained rdkit package. Do not use rdkit-pypi.
run_pip(["install", "-q", "rdkit"], 
        "Installing RDKit")

# Install DRFP without dependency resolution, because drfp==0.3.6 may still refer to old rdkit-pypi metadata.
run_pip(["install", "-q", "--no-deps", "drfp==0.3.6"], 
        "Installing DRFP without dependencies")

# Verify key imports
import numpy as np
import pandas as pd
import torch
import rdkit
from drfp import DrfpEncoder

# Compatibility patch for DRFP on NumPy 2.x.
# DRFP 0.3.6 internally creates np.array(..., dtype=np.int32) from unsigned 32-bit hash values.
# NumPy 2.x raises OverflowError for values larger than int32 max.
# Returning int64 hash values preserves the same numeric values and avoids the overflow.
def patch_drfp_numpy2_overflow():
    import numpy as _np
    from hashlib import blake2b as _blake2b
    from drfp import DrfpEncoder as _DrfpEncoder

    def _safe_hash(shingling):
        hash_values = []
        for t in shingling:
            if isinstance(t, str):
                t = t.encode("utf-8")
            hash_values.append(int(_blake2b(t, digest_size=4).hexdigest(), 16))
        return _np.asarray(hash_values, dtype=_np.int64)

    _DrfpEncoder.hash = staticmethod(_safe_hash)

patch_drfp_numpy2_overflow()

_ = DrfpEncoder.encode(["CCO>>CC=O"], n_folded_length=128)

print("\nSetup complete.")
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("PyTorch:", torch.__version__)
print("RDKit:", rdkit.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Import packages and define EZHit utilities

In [ ]:
#@title Load EZHit utilities { display-mode: "form" }
import os
import re
import json
import zipfile
import random
import warnings
from pathlib import Path
from typing import Any, Optional, List, Dict, Tuple

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn

import matplotlib.pyplot as plt

from huggingface_hub import hf_hub_download

from drfp import DrfpEncoder

# Compatibility patch for DRFP on NumPy 2.x.
# DRFP 0.3.6 internally creates np.array(..., dtype=np.int32) from unsigned 32-bit hash values.
# NumPy 2.x raises OverflowError for values larger than int32 max.
# Returning int64 hash values preserves the same numeric values and avoids the overflow.
def patch_drfp_numpy2_overflow():
    import numpy as _np
    from hashlib import blake2b as _blake2b
    from drfp import DrfpEncoder as _DrfpEncoder

    def _safe_hash(shingling):
        hash_values = []
        for t in shingling:
            if isinstance(t, str):
                t = t.encode("utf-8")
            hash_values.append(int(_blake2b(t, digest_size=4).hexdigest(), 16))
        return _np.asarray(hash_values, dtype=_np.int64)

    _DrfpEncoder.hash = staticmethod(_safe_hash)

patch_drfp_numpy2_overflow()

from rdkit import Chem, RDLogger
from rdkit.Chem import rdMolDescriptors
from rdkit.Chem.MolStandardize import rdMolStandardize

RDLogger.DisableLog("rdApp.*")

from kan import KAN

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


# -----------------------
# Model
# -----------------------
class HybridPairKAN(nn.Module):
    def __init__(self, esm_dim: int, drfp_dim: int, react_dim: int, hidden: int, dropout: float = 0.0):
        super().__init__()
        self.esm_ln = nn.LayerNorm(esm_dim)
        self.drfp_ln = nn.LayerNorm(drfp_dim)
        self.react_ln = nn.LayerNorm(react_dim)

        in_dim = esm_dim + drfp_dim + react_dim
        self.bottleneck_dim = hidden

        self.compressor = nn.Sequential(
            nn.Linear(in_dim, self.bottleneck_dim),
            nn.LayerNorm(self.bottleneck_dim),
            nn.SiLU(),
            nn.Dropout(dropout)
        )

        self.net = KAN([self.bottleneck_dim, hidden, 1])

    def forward(self, esm, drfp, react, update_grid: bool = False, return_latent: bool = False):
        e_norm = self.esm_ln(esm)
        d_norm = self.drfp_ln(drfp)
        r_norm = self.react_ln(react)

        x = torch.cat([e_norm, d_norm, r_norm], dim=-1)
        latent = self.compressor(x)
        logit = self.net(latent, update_grid=update_grid).squeeze(-1)

        if return_latent:
            return logit, latent
        return logit


# -----------------------
# Metrics without scikit-learn
# -----------------------
def binary_accuracy(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_pred = (np.asarray(y_prob) >= threshold).astype(int)
    return float((y_true == y_pred).mean())


def roc_auc_numpy(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)

    pos = y_true == 1
    neg = y_true == 0
    n_pos = int(pos.sum())
    n_neg = int(neg.sum())

    if n_pos == 0 or n_neg == 0:
        return float("nan")

    order = np.argsort(y_score)
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(y_score) + 1)

    # Average ranks for ties
    sorted_scores = y_score[order]
    i = 0
    while i < len(sorted_scores):
        j = i + 1
        while j < len(sorted_scores) and sorted_scores[j] == sorted_scores[i]:
            j += 1
        if j - i > 1:
            avg_rank = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg_rank
        i = j

    rank_sum_pos = ranks[pos].sum()
    auc = (rank_sum_pos - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg)
    return float(auc)


def average_precision_numpy(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)

    n_pos = int((y_true == 1).sum())
    if n_pos == 0:
        return float("nan")

    order = np.argsort(-y_score)
    y_sorted = y_true[order]

    tp = np.cumsum(y_sorted == 1)
    fp = np.cumsum(y_sorted == 0)
    precision = tp / np.maximum(tp + fp, 1)

    ap = (precision * (y_sorted == 1)).sum() / n_pos
    return float(ap)


def binary_metrics(y_true, y_prob, threshold=0.5):
    return {
        "accuracy": binary_accuracy(y_true, y_prob, threshold=threshold),
        "roc_auc": roc_auc_numpy(y_true, y_prob),
        "auprc": average_precision_numpy(y_true, y_prob),
    }


# -----------------------
# Splitting without scikit-learn
# -----------------------
def random_split_indices(n, val_ratio=0.1, test_ratio=0.1, seed=42):
    rng = np.random.default_rng(seed)
    idx = np.arange(n)
    rng.shuffle(idx)

    n_test = int(round(n * test_ratio))
    n_val = int(round(n * val_ratio))

    test_idx = idx[:n_test] if n_test > 0 else np.array([], dtype=int)
    val_idx = idx[n_test:n_test + n_val]
    train_idx = idx[n_test + n_val:]

    return train_idx, val_idx, test_idx


def stratified_split_indices(labels, val_ratio=0.1, test_ratio=0.1, seed=42):
    labels = np.asarray(labels).astype(int)
    train_parts, val_parts, test_parts = [], [], []

    for lab in np.unique(labels):
        lab_idx = np.where(labels == lab)[0]
        tr, va, te = random_split_indices(len(lab_idx), val_ratio=val_ratio, test_ratio=test_ratio, seed=seed + int(lab))
        train_parts.append(lab_idx[tr])
        val_parts.append(lab_idx[va])
        test_parts.append(lab_idx[te])

    rng = np.random.default_rng(seed)
    train_idx = np.concatenate(train_parts) if train_parts else np.array([], dtype=int)
    val_idx = np.concatenate(val_parts) if val_parts else np.array([], dtype=int)
    test_idx = np.concatenate(test_parts) if test_parts else np.array([], dtype=int)

    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    rng.shuffle(test_idx)

    return train_idx, val_idx, test_idx


def group_split_indices(groups, val_ratio=0.1, test_ratio=0.1, seed=42):
    groups = np.asarray(groups).astype(str)
    unique_groups = np.unique(groups)
    rng = np.random.default_rng(seed)
    rng.shuffle(unique_groups)

    n_groups = len(unique_groups)
    n_test_g = int(round(n_groups * test_ratio))
    n_val_g = int(round(n_groups * val_ratio))

    test_groups = set(unique_groups[:n_test_g]) if n_test_g > 0 else set()
    val_groups = set(unique_groups[n_test_g:n_test_g + n_val_g])
    train_groups = set(unique_groups[n_test_g + n_val_g:])

    train_idx = np.where(np.isin(groups, list(train_groups)))[0]
    val_idx = np.where(np.isin(groups, list(val_groups)))[0]
    test_idx = np.where(np.isin(groups, list(test_groups)))[0] if test_groups else np.array([], dtype=int)

    return train_idx, val_idx, test_idx


def split_dataset(df, split_mode, seq_col, rxn_col, label_col, group_col, val_ratio, test_ratio, seed):
    df = df.copy().reset_index(drop=True)

    if split_mode == "Use existing split column":
        if "split" not in df.columns:
            raise ValueError("No 'split' column was found.")
        split_values = df["split"].astype(str).str.lower()
        train_df = df[split_values == "train"].reset_index(drop=True)
        val_df = df[split_values.isin(["val", "valid", "validation"])].reset_index(drop=True)
        test_df = df[split_values == "test"].reset_index(drop=True)
        if len(train_df) == 0 or len(val_df) == 0:
            raise ValueError("The split column must contain train and val/validation rows.")
        return train_df, val_df, None if len(test_df) == 0 else test_df

    if split_mode == "Group-aware split by reaction":
        groups = df[group_col].astype(str).values if group_col in df.columns else df[rxn_col].astype(str).values
        train_idx, val_idx, test_idx = group_split_indices(groups, val_ratio=val_ratio, test_ratio=test_ratio, seed=seed)
    else:
        if label_col in df.columns and df[label_col].nunique() > 1:
            train_idx, val_idx, test_idx = stratified_split_indices(df[label_col].values, val_ratio=val_ratio, test_ratio=test_ratio, seed=seed)
        else:
            train_idx, val_idx, test_idx = random_split_indices(len(df), val_ratio=val_ratio, test_ratio=test_ratio, seed=seed)

    train_df = df.iloc[train_idx].reset_index(drop=True)
    val_df = df.iloc[val_idx].reset_index(drop=True)
    test_df = df.iloc[test_idx].reset_index(drop=True) if len(test_idx) > 0 else None

    if len(train_df) == 0 or len(val_df) == 0:
        raise ValueError("The split produced empty train or validation data. Adjust split ratios or use a larger dataset.")

    return train_df, val_df, test_df


# -----------------------
# Protein encoder: ProtT5
# -----------------------
PROTT5_MODEL_NAME = "Rostlab/prot_t5_xl_half_uniref50-enc"
_prott5_tokenizer = None
_prott5_model = None

def mean_pool_reps(hidden_states, attention_mask):
    """
    Mean pool ProtT5 representations using the attention mask.
    hidden_states: [B, L, 1024]
    attention_mask: [B, L]
    """
    mask_expanded = attention_mask.unsqueeze(-1).float()
    summed = (hidden_states * mask_expanded).sum(dim=1)
    denom = mask_expanded.sum(dim=1).clamp(min=1e-9)
    return summed / denom


def make_dynamic_batches(lengths, max_tokens):
    """
    Length-sorted dynamic batching.

    lengths: estimated token lengths per sequence, including EOS.
    max_tokens: approximate token budget per batch.
    """
    order = np.argsort(-np.asarray(lengths))
    batches = []
    cur = []
    cur_tok = 0

    for idx in order.tolist():
        L = int(lengths[idx])
        if not cur:
            cur = [idx]
            cur_tok = L
            continue

        if cur_tok + L <= max_tokens:
            cur.append(idx)
            cur_tok += L
        else:
            batches.append(cur)
            cur = [idx]
            cur_tok = L

    if cur:
        batches.append(cur)

    return batches


def preprocess_seq_for_prott5(seq: str, max_len: int = 1022):
    """
    ProtT5 sequence preprocessing:
    - truncate to max_len
    - replace rare amino acids U/Z/O/B with X
    - insert spaces between residues
    """
    seq = str(seq).strip()[:max_len]
    seq = re.sub(r"[UZOBuzob]", "X", seq.upper())
    return " ".join(list(seq))


def choose_prott5_dtype():
    """
    Use bf16 on Ampere or newer GPUs; otherwise use fp16 on CUDA.
    T4 does not support efficient bf16, so it will use fp16.
    """
    if not torch.cuda.is_available():
        return torch.float32

    try:
        major, minor = torch.cuda.get_device_capability(0)
        if major >= 8:
            return torch.bfloat16
    except Exception:
        pass

    return torch.float16


def load_prott5(model_name=PROTT5_MODEL_NAME):
    global _prott5_tokenizer, _prott5_model

    if _prott5_tokenizer is not None and _prott5_model is not None:
        return _prott5_tokenizer, _prott5_model

    from transformers import T5Tokenizer, T5EncoderModel

    print(f"[ProtT5] Loading tokenizer and model: {model_name}")
    _prott5_tokenizer = T5Tokenizer.from_pretrained(model_name, do_lower_case=False)

    dtype = choose_prott5_dtype()
    if torch.cuda.is_available():
        _prott5_model = T5EncoderModel.from_pretrained(model_name, torch_dtype=dtype).to(device)
    else:
        _prott5_model = T5EncoderModel.from_pretrained(model_name).to(device)

    _prott5_model.eval()
    print(f"[ProtT5] Model loaded with dtype={dtype}")
    return _prott5_tokenizer, _prott5_model


@torch.no_grad()
def extract_unique_embeddings_dynamic_prott5(
    uniq_seqs,
    model_name=PROTT5_MODEL_NAME,
    max_len=1022,
    max_tokens=4000,
    cache_path=None,
):
    """
    Extract ProtT5 embeddings for unique sequences using length-sorted dynamic batching.
    This follows the same general strategy as the standalone ProtT5 cache script used for EZHit.
    """
    if cache_path and os.path.exists(cache_path):
        print(f"[ProtT5] Loading unique embedding cache: {cache_path}")
        obj = torch.load(cache_path, map_location="cpu")
        if torch.is_tensor(obj):
            return obj.float()
        raise ValueError(f"Unsupported ProtT5 unique cache format: {cache_path}")

    tokenizer, model = load_prott5(model_name=model_name)

    processed_seqs = [preprocess_seq_for_prott5(s, max_len=max_len) for s in uniq_seqs]
    lengths = [min(len(str(s)), max_len) + 1 for s in uniq_seqs]
    batches = make_dynamic_batches(lengths, max_tokens=max_tokens)

    print(f"[ProtT5] unique={len(uniq_seqs):,} | batches={len(batches):,} | max_tokens={max_tokens}")

    out = None
    dtype = choose_prott5_dtype()

    for b in tqdm(batches, desc="[ProtT5] Dynamic batches"):
        batch_seqs = [processed_seqs[i] for i in b]

        inputs = tokenizer(
            batch_seqs,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_len + 1
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        if torch.cuda.is_available():
            with torch.autocast("cuda", dtype=dtype):
                outputs = model(**inputs)
                pooled = mean_pool_reps(outputs.last_hidden_state, inputs["attention_mask"])
        else:
            outputs = model(**inputs)
            pooled = mean_pool_reps(outputs.last_hidden_state, inputs["attention_mask"])

        if out is None:
            dim = int(pooled.shape[1])
            out = torch.zeros((len(uniq_seqs), dim), dtype=torch.float32)

        out[b] = pooled.detach().cpu().float()

    if out is None:
        out = torch.zeros((len(uniq_seqs), 1024), dtype=torch.float32)

    if cache_path:
        os.makedirs(os.path.dirname(cache_path) or ".", exist_ok=True)
        torch.save(out, cache_path)
        print(f"[ProtT5] Saved unique embedding cache: {cache_path}")

    return out


def build_row_aligned_protein_embeddings(
    df,
    seq_col,
    cache_path=None,
    max_len=1022,
    max_tokens=4000,
    model_name=PROTT5_MODEL_NAME,
    output_dtype="float32",
):
    """
    Build row-aligned ProtT5 embeddings with:
    - sequence deduplication
    - dynamic batching by approximate token budget
    - row-aligned reconstruction

    The standalone training script can save float16 tensors for storage.
    Here the tensor is returned as float32 by default for stable fine-tuning.
    """
    if cache_path and os.path.exists(cache_path):
        print(f"[ProtT5] Loading row-aligned protein embedding cache: {cache_path}")
        obj = torch.load(cache_path, map_location="cpu")
        if torch.is_tensor(obj):
            return obj.float()
        raise ValueError(f"Unsupported row-aligned protein cache format: {cache_path}")

    seqs = df[seq_col].fillna("").astype(str).tolist()
    n = len(seqs)

    uniq_seqs = []
    seq2uid = {}
    uid_of_row = np.empty((n,), dtype=np.int64)

    for i, s in enumerate(seqs):
        if s not in seq2uid:
            seq2uid[s] = len(uniq_seqs)
            uniq_seqs.append(s)
        uid_of_row[i] = seq2uid[s]

    print(f"[ProtT5] rows={n:,} unique sequences={len(uniq_seqs):,} (ratio={len(uniq_seqs)/max(1,n):.4f})")

    unique_cache = cache_path.replace(".pt", "_unique.pt") if cache_path else None

    embs_unique = extract_unique_embeddings_dynamic_prott5(
        uniq_seqs=uniq_seqs,
        model_name=model_name,
        max_len=max_len,
        max_tokens=max_tokens,
        cache_path=unique_cache,
    )

    emb = embs_unique[uid_of_row].contiguous()

    if output_dtype == "float16":
        emb_to_save = emb.half()
    else:
        emb_to_save = emb.float()

    if cache_path:
        os.makedirs(os.path.dirname(cache_path) or ".", exist_ok=True)
        torch.save(emb_to_save, cache_path)
        print(f"[ProtT5] Saved row-aligned protein embeddings: {cache_path}")
        print(f"[ProtT5] Tensor shape: {tuple(emb_to_save.shape)}, dtype: {emb_to_save.dtype}")

    return emb.float()

# -----------------------
# Reaction features
# -----------------------
def l2norm_np(x, eps=1e-12):
    n = np.linalg.norm(x, axis=-1, keepdims=True)
    return x / np.maximum(n, eps)


def standardize_reaction_smiles(
    rxn_smiles: str,
    preserve_direction: bool = False,
    uncharge: bool = False,
    filter_charged_single_atom: bool = False,
):
    """
    RDKit-based reaction SMILES standardization.

    This follows the same logic as the DRFP preprocessing script used during EZHit training:
    - split the reaction into molecular components
    - canonicalize each molecule using RDKit
    - optionally uncharge molecules
    - optionally remove single charged atoms such as ions
    - optionally preserve the reactants>>products direction

    Stereochemistry is preserved through isomericSmiles=True.
    """
    if rxn_smiles is None or not isinstance(rxn_smiles, str):
        return None
    if ">>" not in rxn_smiles:
        return None

    uncharger = rdMolStandardize.Uncharger() if uncharge else None
    sides = rxn_smiles.split(">>")
    new_parts = []

    for side in sides:
        mols = []
        for s in side.split("."):
            s = s.strip()
            if not s:
                continue
            try:
                m = Chem.MolFromSmiles(s)
                if m is None:
                    continue

                if filter_charged_single_atom and m.GetNumAtoms() == 1:
                    atom = m.GetAtomWithIdx(0)
                    if atom.GetFormalCharge() != 0:
                        continue

                if uncharger is not None:
                    try:
                        m = uncharger.uncharge(m)
                    except Exception:
                        pass

                smi = Chem.MolToSmiles(m, canonical=True, isomericSmiles=True)
                if smi:
                    mols.append(smi)
            except Exception:
                continue

        mols.sort()
        new_parts.append(".".join(mols))

    if len(new_parts) < 2 or not new_parts[0] or not new_parts[1]:
        return None

    if not preserve_direction:
        new_parts.sort()

    return ">>".join(new_parts)


def drfp_encode(smiles_list, n_folded_length=2048):
    fps = DrfpEncoder.encode(smiles_list, n_folded_length=n_folded_length)
    return np.asarray(fps, dtype=np.float32)


def make_drfp_rowaligned_dedup(
    df,
    rxn_col,
    fp_dim=2048,
    standardize_smiles=True,
    preserve_direction=False,
    uncharge=False,
    filter_charged_single_atom=False,
    failure_mode="strict",
    report_path=None,
):
    """
    Build row-aligned DRFP features using the same standardization logic as the training script.

    Compared with the previous "safe zero-vector" behavior, the default mode here is strict:
    if any reaction still cannot be encoded after optional standardization, the notebook stops
    and writes a report listing the problematic reactions.

    failure_mode:
      - "strict": stop if DRFP encoding fails for any unique reaction
      - "zero_vector": keep running by assigning zero vectors to failed reactions
    """
    original_smis = df[rxn_col].fillna("").astype(str).tolist()
    used_smis = []
    std_status = []

    if standardize_smiles:
        fail = 0
        for s in tqdm(original_smis, desc="[RDKit] Standardize reaction SMILES"):
            ss = standardize_reaction_smiles(
                s,
                preserve_direction=preserve_direction,
                uncharge=uncharge,
                filter_charged_single_atom=filter_charged_single_atom,
            )
            if ss is None:
                used_smis.append("")
                std_status.append("standardization_failed")
                fail += 1
            else:
                used_smis.append(ss)
                std_status.append("standardized")
        print(f"[Diag] DRFP standardization failed: {fail}/{len(original_smis)} ({fail/max(1,len(original_smis)):.2%})")
    else:
        used_smis = original_smis
        std_status = ["original"] * len(original_smis)

    uniq = []
    smi2uid = {}
    uids = np.empty((len(used_smis),), dtype=np.int64)
    uid_first_original = {}
    uid_first_status = {}

    for i, s in enumerate(used_smis):
        uid = smi2uid.get(s)
        if uid is None:
            uid = len(uniq)
            smi2uid[s] = uid
            uniq.append(s)
            uid_first_original[uid] = original_smis[i]
            uid_first_status[uid] = std_status[i]
        uids[i] = uid

    print(f"[DRFP] rows={len(used_smis):,} unique_rxn={len(uniq):,} (ratio={len(uniq)/max(1,len(used_smis)):.4f})")

    fps_u = np.zeros((len(uniq), fp_dim), dtype=np.float32)
    report_rows = []
    failed_rows = []

    chunk = 4096
    for start in tqdm(range(0, len(uniq), chunk), desc="[DRFP] Encode unique"):
        part = uniq[start:start + chunk]

        try:
            fps_u[start:start + len(part)] = drfp_encode(part, n_folded_length=fp_dim)
            for j, s in enumerate(part):
                uid = start + j
                report_rows.append({
                    "uid": uid,
                    "original_rxn_smiles": uid_first_original.get(uid, ""),
                    "used_rxn_smiles": s,
                    "standardization_status": uid_first_status.get(uid, ""),
                    "encoding_status": "ok",
                    "error": "",
                })
        except Exception as batch_error:
            # Diagnose individual reactions so users can fix data instead of guessing.
            print("Batch DRFP encoding failed. Diagnosing reactions one by one...")
            for j, s in enumerate(part):
                uid = start + j
                try:
                    fps_u[uid] = drfp_encode([s], n_folded_length=fp_dim)[0]
                    report_rows.append({
                        "uid": uid,
                        "original_rxn_smiles": uid_first_original.get(uid, ""),
                        "used_rxn_smiles": s,
                        "standardization_status": uid_first_status.get(uid, ""),
                        "encoding_status": "ok_after_individual_retry",
                        "error": "",
                    })
                except Exception as e:
                    err = str(e).replace("\n", " ")[:1000]
                    row = {
                        "uid": uid,
                        "original_rxn_smiles": uid_first_original.get(uid, ""),
                        "used_rxn_smiles": s,
                        "standardization_status": uid_first_status.get(uid, ""),
                        "encoding_status": "failed",
                        "error": err,
                    }
                    report_rows.append(row)
                    failed_rows.append(row)

                    if failure_mode == "zero_vector":
                        fps_u[uid] = np.zeros((fp_dim,), dtype=np.float32)

    if report_path:
        os.makedirs(os.path.dirname(report_path) or ".", exist_ok=True)
        pd.DataFrame(report_rows).to_csv(report_path, index=False)
        print(f"Saved DRFP encoding report: {report_path}")

    if failed_rows:
        failed_df = pd.DataFrame(failed_rows)
        print("\nDRFP failed reactions:")
        display(failed_df[["uid", "standardization_status", "original_rxn_smiles", "used_rxn_smiles", "error"]].head(20))

        if failure_mode == "strict":
            raise RuntimeError(
                f"DRFP failed for {len(failed_rows)} unique reaction(s). "
                f"A report was saved to: {report_path}. "
                "Please inspect/fix these reaction SMILES, or switch failure_mode to zero_vector for quick testing."
            )
        else:
            print(
                f"Warning: {len(failed_rows)} unique reaction(s) failed DRFP encoding and were assigned zero vectors. "
                "This mode is intended for quick testing, not formal evaluation."
            )

    fps_u = l2norm_np(fps_u)
    fps = fps_u[uids]
    return torch.tensor(fps, dtype=torch.float32)
def get_reactant_morgan_fp(rxn_smiles, radius=2, nBits=2048):
    if not isinstance(rxn_smiles, str) or ">>" not in rxn_smiles:
        return np.zeros((nBits,), dtype=np.float32)

    reactants_smi = rxn_smiles.split(">>")[0]
    mol = Chem.MolFromSmiles(reactants_smi)

    if mol is None:
        return np.zeros((nBits,), dtype=np.float32)

    fp = rdMolDescriptors.GetMorganFingerprintAsBitVect(mol, radius, nBits=nBits)
    arr = np.zeros((nBits,), dtype=np.float32)
    Chem.DataStructs.ConvertToNumpyArray(fp, arr)
    return arr


def make_reactant_morgan_rowaligned_dedup(df, rxn_col, fp_dim=2048):
    rxns = df[rxn_col].fillna("").astype(str).tolist()

    uniq = []
    rxn_to_uid = {}
    row_uids = np.empty(len(rxns), dtype=np.int64)

    for i, rxn in enumerate(rxns):
        if rxn not in rxn_to_uid:
            rxn_to_uid[rxn] = len(uniq)
            uniq.append(rxn)
        row_uids[i] = rxn_to_uid[rxn]

    print(f"Reaction rows={len(rxns):,}; unique reactions for reactant Morgan FP={len(uniq):,}")

    fps_u = np.zeros((len(uniq), fp_dim), dtype=np.float32)

    for i, rxn in enumerate(tqdm(uniq, desc="Encoding reactant Morgan FP")):
        fps_u[i] = get_reactant_morgan_fp(rxn, radius=2, nBits=fp_dim)

    fps_u = l2norm_np(fps_u)
    return torch.tensor(fps_u[row_uids], dtype=torch.float32)


def load_or_build_tensor(cache_path, builder_fn):
    if cache_path and os.path.exists(cache_path):
        print(f"Loading cache: {cache_path}")
        obj = torch.load(cache_path, map_location="cpu")
        if not torch.is_tensor(obj):
            raise ValueError(f"Cache does not contain a tensor: {cache_path}")
        return obj.float()

    tensor = builder_fn()

    if cache_path:
        os.makedirs(os.path.dirname(cache_path) or ".", exist_ok=True)
        torch.save(tensor, cache_path)
        print(f"Saved cache: {cache_path}")

    return tensor


# -----------------------
# Dataset utilities
# -----------------------
def validate_and_clean_dataset(df, seq_col, rxn_col, label_col):
    missing = [c for c in [seq_col, rxn_col] if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df = df.copy()
    df[seq_col] = df[seq_col].fillna("").astype(str).str.strip()
    df[rxn_col] = df[rxn_col].fillna("").astype(str).str.strip()

    df = df[(df[seq_col] != "") & (df[rxn_col] != "")]
    df = df[df[rxn_col].str.contains(">>", regex=False)]

    if label_col in df.columns:
        df[label_col] = df[label_col].astype(int)

    return df.reset_index(drop=True)


def generate_random_negative_samples(df, seq_col, rxn_col, label_col="Label", ratio=1.0, seed=42):
    rng = np.random.default_rng(seed)

    if label_col not in df.columns:
        df = df.copy()
        df[label_col] = 1

    pos_df = df[df[label_col] == 1].copy()
    if len(pos_df) == 0:
        raise ValueError("No positive samples found. Negative sampling requires positive pairs.")

    n_neg_target = int(round(len(pos_df) * ratio))
    if n_neg_target <= 0:
        return df.copy()

    seqs = pos_df[seq_col].dropna().astype(str).unique().tolist()
    rxns = pos_df[rxn_col].dropna().astype(str).unique().tolist()

    existing_pos = set(zip(pos_df[seq_col].astype(str), pos_df[rxn_col].astype(str)))
    neg_pairs = set()
    neg_rows = []

    max_trials = max(n_neg_target * 100, 1000)
    trials = 0

    while len(neg_rows) < n_neg_target and trials < max_trials:
        trials += 1
        seq = str(rng.choice(seqs))
        rxn = str(rng.choice(rxns))
        pair = (seq, rxn)

        if pair in existing_pos or pair in neg_pairs:
            continue

        neg_pairs.add(pair)
        neg_rows.append({seq_col: seq, rxn_col: rxn, label_col: 0})

    if len(neg_rows) < n_neg_target:
        print(f"Warning: generated {len(neg_rows):,}/{n_neg_target:,} requested negatives.")

    neg_df = pd.DataFrame(neg_rows)
    common_cols = [c for c in df.columns if c in neg_df.columns]
    for c in [seq_col, rxn_col, label_col]:
        if c not in common_cols:
            common_cols.append(c)

    out = pd.concat([df, neg_df[common_cols]], ignore_index=True, sort=False)
    out[label_col] = out[label_col].astype(int)
    out = out.drop_duplicates(subset=[seq_col, rxn_col, label_col]).reset_index(drop=True)

    print("Label distribution after negative sampling:")
    print(out[label_col].value_counts())

    return out


# -----------------------
# Training helpers
# -----------------------
@torch.no_grad()
def predict_probs_and_loss(model, esm, drfp, react, labels, criterion, batch_size=1024):
    model.eval()
    probs = []
    losses = []
    y_tensor = torch.tensor(labels.astype(np.float32), dtype=torch.float32)

    for i in range(0, len(labels), batch_size):
        e = esm[i:i + batch_size].to(device)
        f = drfp[i:i + batch_size].to(device)
        r = react[i:i + batch_size].to(device)
        y = y_tensor[i:i + batch_size].to(device)

        logit = model(e, f, r)
        loss = criterion(logit, y)
        prob = torch.sigmoid(logit)

        probs.append(prob.detach().cpu())
        losses.append(float(loss.detach().cpu().item()))

    return torch.cat(probs).numpy().astype(np.float32), float(np.mean(losses))


@torch.no_grad()
def collect_latents(model, esm, drfp, react, labels, positive_only=True, batch_size=1024):
    model.eval()

    if positive_only:
        idx = np.where(labels.astype(int) == 1)[0]
        if len(idx) == 0:
            raise ValueError("No positive samples are available for Mahalanobis statistics.")
    else:
        idx = np.arange(len(labels))

    latents = []

    for i in range(0, len(idx), batch_size):
        b = idx[i:i + batch_size]
        e = esm[b].to(device)
        f = drfp[b].to(device)
        r = react[b].to(device)
        _, latent = model(e, f, r, return_latent=True)
        latents.append(latent.detach().cpu().float())

    return torch.cat(latents, dim=0)


def save_mahalanobis_stats(model, esm_train, drfp_train, react_train, y_train, out_path, shrinkage=1e-3):
    latents = collect_latents(model, esm_train, drfp_train, react_train, y_train, positive_only=True)
    n, d = latents.shape
    mu = latents.mean(dim=0)
    centered = latents - mu.view(1, -1)

    if n > 1:
        cov = centered.T @ centered / max(1, n - 1)
    else:
        cov = torch.eye(d)

    cov = cov + shrinkage * torch.eye(d)
    inv_cov = torch.linalg.pinv(cov)

    stat = {
        "mean": mu.float(),
        "inv_cov": inv_cov.float(),
        "n_positive": int(n),
        "hidden_dim": int(d),
        "shrinkage": float(shrinkage),
        "description": "Positive-class latent distribution statistics for EZHit Mahalanobis-distance inference."
    }

    torch.save(stat, out_path)
    print(f"Saved Mahalanobis statistics: {out_path}")
    print(f"mean shape={tuple(mu.shape)}, inv_cov shape={tuple(inv_cov.shape)}, n_positive={n}")


def plot_training_log(log_df, out_dir):
    os.makedirs(out_dir, exist_ok=True)

    plt.figure()
    plt.plot(log_df["epoch"], log_df["train_loss"], label="train_loss")
    plt.plot(log_df["epoch"], log_df["val_loss"], label="val_loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.title("Training and validation loss")
    loss_png = os.path.join(out_dir, "loss_curve.png")
    plt.savefig(loss_png, dpi=200, bbox_inches="tight")
    plt.show()

    plt.figure()
    plt.plot(log_df["epoch"], log_df["train_accuracy"], label="train_accuracy")
    plt.plot(log_df["epoch"], log_df["val_accuracy"], label="val_accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.title("Training and validation accuracy")
    acc_png = os.path.join(out_dir, "accuracy_curve.png")
    plt.savefig(acc_png, dpi=200, bbox_inches="tight")
    plt.show()

    print(f"Saved plots: {loss_png}, {acc_png}")


def load_checkpoint_from_hf(repo_id, model_group, seed, custom_filename=""):
    group_to_file = {
        "General (Pan-Enzyme)": f"checkpoints/binarycls_best_val_seed{seed}.pt",
        "Cytochrome P450": f"checkpoints/ft_p450_best_seed{seed}.pt",
        "Phosphatase": f"checkpoints/ft_phosphatase_best_seed{seed}.pt",
        "Terpene Synthase": f"checkpoints/ft_terpene_best_seed{seed}.pt",
    }

    filename = custom_filename.strip() if custom_filename and custom_filename.strip() else group_to_file[model_group]
    print(f"Downloading checkpoint: {repo_id}/{filename}")
    ckpt_path = hf_hub_download(repo_id=repo_id, filename=filename)
    print(f"Checkpoint path: {ckpt_path}")
    return ckpt_path

## 3. Interactive configuration

Run this cell, choose settings, then click **Load dataset CSV**.

In [ ]:
#@title Interactive configuration { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files

repo_id_w = widgets.Text(value="deanluo/EzHit", description="HF repo:", layout=widgets.Layout(width="600px"))
model_group_w = widgets.Dropdown(
    options=["General (Pan-Enzyme)", "Cytochrome P450", "Phosphatase", "Terpene Synthase"],
    value="General (Pan-Enzyme)",
    description="Model:"
)
seed_w = widgets.Dropdown(options=[40, 41, 42, 43, 44], value=40, description="Seed:")
custom_ckpt_w = widgets.Text(value="", description="Custom ckpt:", placeholder="Optional: checkpoints/my_model.pt", layout=widgets.Layout(width="600px"))

seq_col_w = widgets.Text(value="protein_sequence", description="Seq col:")
rxn_col_w = widgets.Text(value="CANO_RXN_SMILES", description="Rxn col:")
label_col_w = widgets.Text(value="Label", description="Label col:")
group_col_w = widgets.Text(value="CANO_RXN_SMILES", description="Group col:")

generate_neg_w = widgets.Checkbox(value=False, description="Generate random negative samples")
neg_ratio_w = widgets.FloatSlider(value=1.0, min=0.1, max=10.0, step=0.1, description="Neg ratio:")


standardize_drfp_w = widgets.Checkbox(value=True, description="Standardize reaction SMILES for DRFP")
preserve_direction_w = widgets.Checkbox(value=False, description="Preserve reaction direction during DRFP standardization")
uncharge_w = widgets.Checkbox(value=False, description="Uncharge molecules during standardization")
filter_ions_w = widgets.Checkbox(value=False, description="Filter single charged atoms")
drfp_failure_w = widgets.Dropdown(
    options=["strict", "zero_vector"],
    value="strict",
    description="DRFP failure:",
    layout=widgets.Layout(width="350px")
)

split_mode_w = widgets.Dropdown(
    options=["Use existing split column", "Group-aware split by reaction", "Random stratified split"],
    value="Group-aware split by reaction",
    description="Split mode:",
    layout=widgets.Layout(width="450px")
)
val_ratio_w = widgets.FloatSlider(value=0.10, min=0.05, max=0.40, step=0.01, description="Val ratio:")
test_ratio_w = widgets.FloatSlider(value=0.10, min=0.00, max=0.40, step=0.01, description="Test ratio:")

epochs_w = widgets.IntSlider(value=10, min=1, max=100, step=1, description="Epochs:")
batch_size_w = widgets.Dropdown(options=[32, 64, 128, 256, 512], value=128, description="Batch:")
prot_max_tokens_w = widgets.IntSlider(value=4000, min=1000, max=12000, step=500, description="ProtT5 max tokens:", layout=widgets.Layout(width="500px"))
prot_max_len_w = widgets.IntSlider(value=1022, min=128, max=1022, step=1, description="ProtT5 max len:", layout=widgets.Layout(width="500px"))
prot_cache_dtype_w = widgets.Dropdown(options=["float32", "float16"], value="float32", description="Protein cache dtype:")

lr_w = widgets.FloatText(value=1e-4, description="LR:")
weight_decay_w = widgets.FloatText(value=1e-4, description="WD:")
dropout_w = widgets.FloatSlider(value=0.1, min=0.0, max=0.5, step=0.05, description="Dropout:")
patience_w = widgets.IntSlider(value=3, min=1, max=20, step=1, description="Patience:")
train_seed_w = widgets.IntText(value=42, description="Train seed:")

select_metric_w = widgets.Dropdown(
    options=["val_roc_auc", "val_auprc", "val_accuracy", "val_loss"],
    value="val_roc_auc",
    description="Select metric:",
    layout=widgets.Layout(width="350px")
)

output_dir_w = widgets.Text(value="/content/ezhit_finetune_outputs", description="Output:", layout=widgets.Layout(width="600px"))

upload_button = widgets.Button(description="Load dataset CSV", button_style="primary")
dataset_status = widgets.Output()
DATASET_PATH = None

def on_upload_clicked(b):
    global DATASET_PATH
    with dataset_status:
        clear_output()
        print("Please choose a CSV dataset file...")
        uploaded = files.upload()
        if not uploaded:
            print("No file uploaded.")
            return
        fname = list(uploaded.keys())[0]
        DATASET_PATH = f"/content/{fname}"
        print(f"Loaded dataset: {DATASET_PATH}")

upload_button.on_click(on_upload_clicked)

display(widgets.HTML("<h3>Checkpoint</h3>"))
display(repo_id_w, model_group_w, seed_w, custom_ckpt_w)

display(widgets.HTML("<h3>Dataset columns</h3>"))
display(seq_col_w, rxn_col_w, label_col_w, group_col_w)


display(widgets.HTML("<h3>DRFP preprocessing</h3>"))
display(standardize_drfp_w, preserve_direction_w, uncharge_w, filter_ions_w, drfp_failure_w)

display(widgets.HTML("<h3>Negative sampling</h3>"))
display(generate_neg_w, neg_ratio_w)

display(widgets.HTML("<h3>Data split</h3>"))
display(split_mode_w, val_ratio_w, test_ratio_w)

display(widgets.HTML("<h3>Training</h3>"))
display(epochs_w, batch_size_w, prot_max_tokens_w, prot_max_len_w, prot_cache_dtype_w, lr_w, weight_decay_w, dropout_w, patience_w, train_seed_w, select_metric_w, output_dir_w)

display(upload_button, dataset_status)

## 4. Run fine-tuning

In [ ]:
#@title Run fine-tuning { display-mode: "form" }
def run_ezhit_finetuning():
    global DATASET_PATH

    if DATASET_PATH is None or not os.path.exists(DATASET_PATH):
        raise RuntimeError("Please click 'Load dataset CSV' and upload a dataset before running fine-tuning.")

    repo_id = repo_id_w.value.strip()
    model_group = model_group_w.value
    ckpt_seed = int(seed_w.value)
    custom_ckpt = custom_ckpt_w.value.strip()

    seq_col = seq_col_w.value.strip()
    rxn_col = rxn_col_w.value.strip()
    label_col = label_col_w.value.strip()
    group_col = group_col_w.value.strip()

    standardize_drfp = bool(standardize_drfp_w.value)
    preserve_direction = bool(preserve_direction_w.value)
    uncharge = bool(uncharge_w.value)
    filter_charged_single_atom = bool(filter_ions_w.value)
    drfp_failure_mode = drfp_failure_w.value

    generate_negatives = bool(generate_neg_w.value)
    negative_ratio = float(neg_ratio_w.value)

    split_mode = split_mode_w.value
    val_ratio = float(val_ratio_w.value)
    test_ratio = float(test_ratio_w.value)

    epochs = int(epochs_w.value)
    batch_size = int(batch_size_w.value)
    prot_max_tokens = int(prot_max_tokens_w.value)
    prot_max_len = int(prot_max_len_w.value)
    prot_cache_dtype = prot_cache_dtype_w.value
    lr = float(lr_w.value)
    weight_decay = float(weight_decay_w.value)
    dropout = float(dropout_w.value)
    patience = int(patience_w.value)
    train_seed = int(train_seed_w.value)
    select_metric = select_metric_w.value
    out_dir = output_dir_w.value.strip()

    os.makedirs(out_dir, exist_ok=True)
    cache_dir = os.path.join(out_dir, "cache")
    os.makedirs(cache_dir, exist_ok=True)

    random.seed(train_seed)
    np.random.seed(train_seed)
    torch.manual_seed(train_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(train_seed)

    print("=" * 80)
    print("EZHit fine-tuning started")
    print("=" * 80)

    df = pd.read_csv(DATASET_PATH, low_memory=False)
    df = validate_and_clean_dataset(df, seq_col=seq_col, rxn_col=rxn_col, label_col=label_col)
    print(f"Dataset loaded: {DATASET_PATH}")
    print(f"Rows after cleaning: {len(df):,}")

    if generate_negatives:
        print("\nGenerating random negative samples...")
        df = generate_random_negative_samples(
            df=df,
            seq_col=seq_col,
            rxn_col=rxn_col,
            label_col=label_col,
            ratio=negative_ratio,
            seed=train_seed
        )
    else:
        if label_col not in df.columns:
            raise ValueError(f"Label column '{label_col}' was not found. Enable negative sampling or provide labels.")
        df[label_col] = df[label_col].astype(int)

    print("\nLabel distribution:")
    print(df[label_col].value_counts())

    print("\nSplitting dataset...")
    train_df, val_df, test_df = split_dataset(
        df=df,
        split_mode=split_mode,
        seq_col=seq_col,
        rxn_col=rxn_col,
        label_col=label_col,
        group_col=group_col,
        val_ratio=val_ratio,
        test_ratio=test_ratio,
        seed=train_seed
    )

    print(f"Train rows: {len(train_df):,}")
    print(f"Val rows:   {len(val_df):,}")
    print(f"Test rows:  {0 if test_df is None else len(test_df):,}")

    train_df.to_csv(os.path.join(out_dir, "train_split.csv"), index=False)
    val_df.to_csv(os.path.join(out_dir, "val_split.csv"), index=False)
    if test_df is not None:
        test_df.to_csv(os.path.join(out_dir, "test_split.csv"), index=False)

    ckpt_path = load_checkpoint_from_hf(repo_id=repo_id, model_group=model_group, seed=ckpt_seed, custom_filename=custom_ckpt)

    print("\nBuilding protein embeddings and reaction features...")
    esm_train = build_row_aligned_protein_embeddings(train_df, seq_col=seq_col, cache_path=os.path.join(cache_dir, "train_protein.pt"), max_len=prot_max_len, max_tokens=prot_max_tokens, output_dtype=prot_cache_dtype)
    esm_val = build_row_aligned_protein_embeddings(val_df, seq_col=seq_col, cache_path=os.path.join(cache_dir, "val_protein.pt"), max_len=prot_max_len, max_tokens=prot_max_tokens, output_dtype=prot_cache_dtype)
    esm_test = build_row_aligned_protein_embeddings(test_df, seq_col=seq_col, cache_path=os.path.join(cache_dir, "test_protein.pt"), max_len=prot_max_len, max_tokens=prot_max_tokens, output_dtype=prot_cache_dtype) if test_df is not None else None

    drfp_train = load_or_build_tensor(os.path.join(cache_dir, "train_drfp.pt"), lambda: make_drfp_rowaligned_dedup(
            train_df, rxn_col=rxn_col, fp_dim=2048,
            standardize_smiles=standardize_drfp,
            preserve_direction=preserve_direction,
            uncharge=uncharge,
            filter_charged_single_atom=filter_charged_single_atom,
            failure_mode=drfp_failure_mode,
            report_path=os.path.join(out_dir, "drfp_train_report.csv")
        ))
    drfp_val = load_or_build_tensor(os.path.join(cache_dir, "val_drfp.pt"), lambda: make_drfp_rowaligned_dedup(
            val_df, rxn_col=rxn_col, fp_dim=2048,
            standardize_smiles=standardize_drfp,
            preserve_direction=preserve_direction,
            uncharge=uncharge,
            filter_charged_single_atom=filter_charged_single_atom,
            failure_mode=drfp_failure_mode,
            report_path=os.path.join(out_dir, "drfp_val_report.csv")
        ))
    drfp_test = load_or_build_tensor(os.path.join(cache_dir, "test_drfp.pt"), lambda: make_drfp_rowaligned_dedup(
            test_df, rxn_col=rxn_col, fp_dim=2048,
            standardize_smiles=standardize_drfp,
            preserve_direction=preserve_direction,
            uncharge=uncharge,
            filter_charged_single_atom=filter_charged_single_atom,
            failure_mode=drfp_failure_mode,
            report_path=os.path.join(out_dir, "drfp_test_report.csv")
        )) if test_df is not None else None

    react_train = load_or_build_tensor(os.path.join(cache_dir, "train_reactant.pt"), lambda: make_reactant_morgan_rowaligned_dedup(train_df, rxn_col=rxn_col, fp_dim=2048))
    react_val = load_or_build_tensor(os.path.join(cache_dir, "val_reactant.pt"), lambda: make_reactant_morgan_rowaligned_dedup(val_df, rxn_col=rxn_col, fp_dim=2048))
    react_test = load_or_build_tensor(os.path.join(cache_dir, "test_reactant.pt"), lambda: make_reactant_morgan_rowaligned_dedup(test_df, rxn_col=rxn_col, fp_dim=2048)) if test_df is not None else None

    y_train = train_df[label_col].to_numpy(dtype=np.float32)
    y_val = val_df[label_col].to_numpy(dtype=np.float32)
    y_test = test_df[label_col].to_numpy(dtype=np.float32) if test_df is not None else None

    ckpt = torch.load(ckpt_path, map_location="cpu")
    ckpt_args = ckpt.get("args", {}) if isinstance(ckpt, dict) else {}
    hidden = int(ckpt_args.get("hidden", 512))

    print(f"\nInitializing model with hidden={hidden}")
    model = HybridPairKAN(
        esm_dim=esm_train.shape[1],
        drfp_dim=drfp_train.shape[1],
        react_dim=react_train.shape[1],
        hidden=hidden,
        dropout=dropout
    ).to(device)

    state = ckpt.get("model", ckpt) if isinstance(ckpt, dict) else ckpt
    model.load_state_dict(state, strict=True)
    print("Loaded pretrained checkpoint.")

    n_pos = int((y_train == 1).sum())
    n_neg = int((y_train == 0).sum())
    pos_weight_val = n_neg / max(1, n_pos)
    print(f"Training positives={n_pos:,}; negatives={n_neg:,}; pos_weight={pos_weight_val:.4f}")

    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight_val], dtype=torch.float32, device=device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    y_train_t = torch.tensor(y_train, dtype=torch.float32)

    best_value = np.inf if select_metric == "val_loss" else -np.inf
    best_epoch = -1
    bad_epochs = 0
    log_rows = []

    safe_group = model_group.replace(" ", "_").replace("(", "").replace(")", "")
    best_ckpt_path = os.path.join(out_dir, f"ezhit_finetuned_{safe_group}_seed{train_seed}.pt")

    print("\nStarting training...")
    for epoch in range(1, epochs + 1):
        model.train()
        perm = np.random.permutation(len(train_df))
        train_losses = []

        for i in tqdm(range(0, len(perm), batch_size), desc=f"Epoch {epoch}/{epochs}", leave=False):
            b = perm[i:i + batch_size]
            e = esm_train[b].to(device)
            f = drfp_train[b].to(device)
            r = react_train[b].to(device)
            y = y_train_t[b].to(device)

            logit = model(e, f, r)
            loss = criterion(logit, y)

            if hasattr(model.net, "regularization_loss"):
                try:
                    loss = loss + 1e-4 * model.net.regularization_loss()
                except Exception:
                    pass

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_losses.append(float(loss.detach().cpu().item()))

        train_probs, train_eval_loss = predict_probs_and_loss(model, esm_train, drfp_train, react_train, y_train, criterion, batch_size=1024)
        val_probs, val_loss = predict_probs_and_loss(model, esm_val, drfp_val, react_val, y_val, criterion, batch_size=1024)

        train_m = binary_metrics(y_train, train_probs)
        val_m = binary_metrics(y_val, val_probs)

        row = {
            "epoch": epoch,
            "train_loss": float(train_eval_loss),
            "train_accuracy": train_m["accuracy"],
            "train_roc_auc": train_m["roc_auc"],
            "train_auprc": train_m["auprc"],
            "val_loss": float(val_loss),
            "val_accuracy": val_m["accuracy"],
            "val_roc_auc": val_m["roc_auc"],
            "val_auprc": val_m["auprc"],
        }
        log_rows.append(row)

        selected_value = row[select_metric]
        improved = selected_value < best_value if select_metric == "val_loss" else ((not np.isnan(selected_value)) and selected_value > best_value)

        print(
            f"Epoch {epoch:03d} | "
            f"train_loss={row['train_loss']:.4f} train_acc={row['train_accuracy']:.4f} | "
            f"val_loss={row['val_loss']:.4f} val_acc={row['val_accuracy']:.4f} "
            f"val_auc={row['val_roc_auc']:.4f} val_auprc={row['val_auprc']:.4f} | "
            f"selected={select_metric}:{selected_value:.4f} improved={improved}"
        )

        if improved:
            best_value = selected_value
            best_epoch = epoch
            bad_epochs = 0
            torch.save(
                {
                    "epoch": epoch,
                    "model": model.state_dict(),
                    "args": {
                        "source_checkpoint": ckpt_path,
                        "source_repo_id": repo_id,
                        "model_group": model_group,
                        "checkpoint_seed": ckpt_seed,
                        "hidden": hidden,
                        "dropout": dropout,
                        "seq_col": seq_col,
                        "rxn_col": rxn_col,
                        "label_col": label_col,
                        "train_seed": train_seed,
                        "lr": lr,
                        "weight_decay": weight_decay,
                        "batch_size": batch_size,
                    },
                    "val_metrics": val_m,
                    "selected_metric": select_metric,
                    "selected_value": float(selected_value),
                },
                best_ckpt_path
            )
            print(f"  Saved best checkpoint: {best_ckpt_path}")
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f"Early stopping at epoch {epoch}.")
                break

    log_df = pd.DataFrame(log_rows)
    log_path = os.path.join(out_dir, "training_log.csv")
    log_df.to_csv(log_path, index=False)
    print(f"\nSaved training log: {log_path}")
    plot_training_log(log_df, out_dir)

    print(f"\nBest epoch: {best_epoch}; best {select_metric}: {best_value:.4f}")

    best_ckpt = torch.load(best_ckpt_path, map_location="cpu")
    model.load_state_dict(best_ckpt["model"])
    model.to(device).eval()

    val_probs, _ = predict_probs_and_loss(model, esm_val, drfp_val, react_val, y_val, criterion, batch_size=1024)
    val_pred_df = val_df.copy()
    val_pred_df["EZHit_probability"] = val_probs
    val_pred_path = os.path.join(out_dir, "val_predictions.csv")
    val_pred_df.to_csv(val_pred_path, index=False)
    print(f"Saved validation predictions: {val_pred_path}")

    if test_df is not None:
        test_probs, test_loss = predict_probs_and_loss(model, esm_test, drfp_test, react_test, y_test, criterion, batch_size=1024)
        test_m = binary_metrics(y_test, test_probs)

        test_pred_df = test_df.copy()
        test_pred_df["EZHit_probability"] = test_probs
        test_pred_path = os.path.join(out_dir, "test_predictions.csv")
        test_pred_df.to_csv(test_pred_path, index=False)

        print("\nTest metrics:")
        print(json.dumps({"test_loss": test_loss, **test_m}, indent=2))
        print(f"Saved test predictions: {test_pred_path}")

    maha_path = os.path.join(out_dir, "train_distribution_stat.pt")
    save_mahalanobis_stats(model, esm_train, drfp_train, react_train, y_train, out_path=maha_path, shrinkage=1e-3)

    config = {
        "repo_id": repo_id,
        "model_group": model_group,
        "checkpoint_seed": ckpt_seed,
        "custom_checkpoint": custom_ckpt,
        "dataset_path": DATASET_PATH,
        "seq_col": seq_col,
        "rxn_col": rxn_col,
        "label_col": label_col,
        "generate_negatives": generate_negatives,
        "standardize_drfp": standardize_drfp,
        "preserve_direction": preserve_direction,
        "uncharge": uncharge,
        "filter_charged_single_atom": filter_charged_single_atom,
        "drfp_failure_mode": drfp_failure_mode,
        "negative_ratio": negative_ratio,
        "split_mode": split_mode,
        "val_ratio": val_ratio,
        "test_ratio": test_ratio,
        "epochs": epochs,
        "batch_size": batch_size,
        "prot_max_tokens": prot_max_tokens,
        "prot_max_len": prot_max_len,
        "prot_cache_dtype": prot_cache_dtype,
        "lr": lr,
        "weight_decay": weight_decay,
        "dropout": dropout,
        "patience": patience,
        "train_seed": train_seed,
        "select_metric": select_metric,
    }

    config_path = os.path.join(out_dir, "run_config.json")
    with open(config_path, "w") as f:
        json.dump(config, f, indent=2)
    print(f"Saved run config: {config_path}")

    zip_path = "/content/ezhit_finetune_outputs.zip"
    if os.path.exists(zip_path):
        os.remove(zip_path)

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, files_ in os.walk(out_dir):
            for fn in files_:
                full = os.path.join(root, fn)
                rel = os.path.relpath(full, out_dir)
                zf.write(full, arcname=rel)

    print("\n" + "=" * 80)
    print("Fine-tuning complete.")
    print(f"Best checkpoint: {best_ckpt_path}")
    print(f"Mahalanobis statistics: {maha_path}")
    print(f"Output ZIP: {zip_path}")
    print("=" * 80)

    try:
        files.download(zip_path)
    except Exception:
        print("Automatic download did not start. Download from the Colab file browser:")
        print(zip_path)


run_ezhit_finetuning()

## 5. Main output files

The most important exported files are:

```text
ezhit_finetuned_*.pt
train_distribution_stat.pt
```

Use the fine-tuned checkpoint for customized inference. Use `train_distribution_stat.pt` with the same checkpoint to enable Mahalanobis-distance inference.